# Test d'Inférence Réelle : `prism-ml/Bonsai-4B` (Hadamard 2-bit complet)

Ce notebook télécharge `prism-ml/Bonsai-4B-gguf`, le convertit en entier au format AI Box (SafeTensors 2-bit + Hadamard $H_{128}$), puis vérifie les vraies sorties texte sur GPU Colab.

In [ ]:
!nvidia-smi
%pip install -q torch safetensors huggingface_hub transformers accelerate gguf

import os
if not os.path.exists("/content/ai-box"):
    !git clone https://github.com/thienban/ai-box.git /content/ai-box
else:
    !git -C /content/ai-box pull origin main

In [ ]:
# Optionnel : si nécessaire pour un dépôt HF privé
# from huggingface_hub import login; login()

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

REPO_GGUF = "prism-ml/Bonsai-4B-gguf"
files = HfApi().list_repo_files(REPO_GGUF)
gguf_files = [f for f in files if f.endswith(".gguf")]
print("Fichiers GGUF trouvés :", gguf_files)

gguf_path = hf_hub_download(repo_id=REPO_GGUF, filename=gguf_files[0])
print("GGUF téléchargé :", gguf_path)

In [ ]:
%cd /content/ai-box
!python3 scripts/convert_gguf_to_aibox.py --gguf "{gguf_path}" --out /content/bonsai-4b-aibox

In [ ]:
%cd /content/ai-box
!python3 scripts/test_bonsai_inference.py --local-dir /content/bonsai-4b-aibox --tokenizer Qwen/Qwen3-4B --max-new-tokens 48